In [ ]:
import time
from IPython.display import display, Javascript

js_code = """
    function ClickConnect() {
        console.log("Keeping Colab session alive by simulating activity...");
        // This simulates a click on the main document body, which helps prevent idle timeouts.
        // It's a common workaround for Colab's idle disconnection.
        document.body.click();
    }
    // Run every 60 seconds (60000 milliseconds)
    setInterval(ClickConnect, 60000);
"""

display(Javascript(js_code))
print("Colab session keep-alive script activated. Check your browser's developer console for messages indicating it's running.")
print("This script runs in your browser and simulates activity to prevent Colab from disconnecting due to inactivity.")
print("You can close this cell's output or stop its execution, but the script will continue running in the background of your browser tab as long as the tab is open.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
image_dir = '/content/drive/MyDrive/XAI_Project/results_multi_cam/Models/'
print(f"Updated image directory to: {image_dir}")

In [ ]:
from pathlib import Path
import numpy as np
import re

# Define base directory (if not already defined in previous cells)
base_results_dir = Path('/content/drive/MyDrive/XAI_Project/results_multi_cam/Models/')

def load_data_for_metrics_corrected(base_dir):
    all_loaded_data = {}
    # Iterate through top-level folders (e.g., efficientnet_b0, LIME Explanations)
    for top_level_folder in base_dir.iterdir():
        if not top_level_folder.is_dir():
            continue

        model_name = top_level_folder.name
        all_loaded_data[model_name] = {}

        # Check if this top_level_folder itself contains .npy files.
        # This covers cases where a "method" like LIME might be a top-level folder
        # containing saliency maps directly, without a sub-method folder.
        npy_files_directly_in_folder = list(top_level_folder.glob('*.npy'))

        if npy_files_directly_in_folder:
            # If .npy files are found directly, treat the folder's name as the method name
            # within this 'model'. This handles "LIME Explanations" as a "model" where
            # the explanations are directly within that folder.
            method_name_for_direct_files = model_name # Use folder name as method name
            all_loaded_data[model_name][method_name_for_direct_files] = {}
            for saliency_map_path in npy_files_directly_in_folder:
                try:
                    match_image_id = re.match(r'(cat_\d+|dog_\d+)', saliency_map_path.name)
                    image_id = match_image_id.group(0) if match_image_id else saliency_map_path.stem.split('_')[0]
                    saliency_map = np.load(saliency_map_path)
                    all_loaded_data[model_name][method_name_for_direct_files][image_id] = saliency_map
                except Exception as e:
                    print(f"⚠️ Error loading {saliency_map_path}: {e}")

        # Now, iterate through sub-folders, which are typically method folders
        for method_folder in top_level_folder.iterdir():
            if method_folder.is_dir(): # Ensure it's a directory (subfolder)
                method_name = method_folder.name
                # Initialize the method_name dictionary if it's not already from direct files
                if method_name not in all_loaded_data[model_name]:
                    all_loaded_data[model_name][method_name] = {}

                # Iterate through ALL .npy files in the method folder
                for saliency_map_path in method_folder.glob('*.npy'): # Changed from *_raw.npy
                    try:
                        match_image_id = re.match(r'(cat_\d+|dog_\d+)', saliency_map_path.name)
                        image_id = match_image_id.group(0) if match_image_id else saliency_map_path.stem.split('_')[0]
                        saliency_map = np.load(saliency_map_path)
                        all_loaded_data[model_name][method_name][image_id] = saliency_map
                    except Exception as e:
                        print(f"⚠️ Error loading {saliency_map_path}: {e}")
    return all_loaded_data

# Call the function to load all data
print(f"Loading saliency map data from: {base_results_dir}")
all_loaded_data = load_data_for_metrics_corrected(base_results_dir)

# Confirmation message
print("\nSaliency map data loaded successfully!")
print(f"Loaded models: {list(all_loaded_data.keys())}")

# Optional: Display a sample of loaded data keys for verification
if all_loaded_data:
    sample_model = list(all_loaded_data.keys())[0]
    if all_loaded_data[sample_model]:
        sample_method = list(all_loaded_data[sample_model].keys())[0]
        print(f"Sample data structure for model '{sample_model}', method '{sample_method}':")
        print(f"  First 5 image IDs: {list(all_loaded_data[sample_model][sample_method].keys())[:5]}")
    else:
        print(f"No methods found for model '{sample_model}'.")
else:
    print("No data was loaded. Please check the `base_results_dir` path and folder structure.")

print("\nVerifying specific methods:")
requested_methods = ['vanilla_saliency', 'smoothgrad', 'integrated_gradients', 'LIME'] # Keep LIME as requested by user
found_requested_methods = {meth: False for meth in requested_methods}

for model_name, methods_data in all_loaded_data.items():
    # Check if the model_name itself matches one of the requested methods (e.g., LIME Explanations)
    for req_meth in requested_methods:
        if req_meth.lower() in model_name.lower() and methods_data: # Check if it's a model that *has* loaded data
            print(f"- Found model '{model_name}' (matching '{req_meth}') with {len(methods_data.keys())} method(s) and first method '{list(methods_data.keys())[0]}' having {len(list(methods_data.values())[0])} images.")
            found_requested_methods[req_meth] = True

    for method_name in methods_data.keys():
        for req_meth in requested_methods:
            if req_meth.lower() in method_name.lower() and methods_data[method_name]:
                print(f"- Found method '{method_name}' under model '{model_name}' (matching '{req_meth}') with {len(methods_data[method_name])} image(s).")
                found_requested_methods[req_meth] = True

for req_meth, found in found_requested_methods.items():
    if not found:
        print(f"- Method '{req_meth}' was NOT found.")

In [ ]:
import itertools
import time
import pandas as pd
import numpy as np

def calculate_dice_coefficient(map1: np.ndarray, map2: np.ndarray) -> float:
    """
    Calculates the Dice Coefficient (Sørensen–Dice coefficient) between two saliency maps.
    Maps are first binarized using their respective medians as thresholds.
    """
    # Ensure maps have the same shape
    if map1.shape != map2.shape:
        print(f"Warning: Maps have different shapes: {map1.shape} vs {map2.shape}. Returning 0.0")
        return 0.0

    # Binarize maps using their respective medians as thresholds
    # Handle constant arrays: if all values are the same, median is that value.
    # Using > median ensures there's at least some '1' if not all values are identical.

    if np.all(map1 == map1.flat[0]): # Check if map1 is constant
        bin_map1 = np.zeros_like(map1, dtype=bool)
    else:
        median_map1 = np.median(map1)
        bin_map1 = map1 > median_map1

    if np.all(map2 == map2.flat[0]): # Check if map2 is constant
        bin_map2 = np.zeros_like(map2, dtype=bool)
    else:
        median_map2 = np.median(map2)
        bin_map2 = map2 > median_map2

    # Calculate Intersection
    intersection = np.sum(np.logical_and(bin_map1, bin_map2))

    # Calculate sum of elements in both binary maps
    sum_of_maps = np.sum(bin_map1) + np.sum(bin_map2)

    if sum_of_maps == 0:
        return 0.0 # Avoid division by zero if both masks are empty
    else:
        return (2.0 * intersection) / sum_of_maps

print("Dice Coefficient calculation function defined.")

start_time_dice = time.time()

dice_all_models_results = []

# Assuming all_loaded_data is available from previous cells
if 'all_loaded_data' not in locals() or not all_loaded_data:
    print("Error: `all_loaded_data` not found. Please run the data loading cell first.")
else:
    # Ensure the output directory exists
    output_dir = Path('/content/drive/MyDrive/XAI_Project/results_multi_cam/results_all/')
    output_dir.mkdir(parents=True, exist_ok=True)

    # Iterate through each model
    for model_name, methods_data in all_loaded_data.items():
        method_names = list(methods_data.keys())

        # Iterate through all pairs of methods, including self-comparison and reversed pairs
        for method_a_name, method_b_name in itertools.product(method_names, repeat=2):
            # Get the set of image IDs common to both methods for this model
            common_image_ids = set(methods_data[method_a_name].keys()).intersection(methods_data[method_b_name].keys())

            for image_id in common_image_ids:
                map_a = methods_data[method_a_name][image_id]
                map_b = methods_data[method_b_name][image_id]

                # Ensure maps have the same shape before calculating Dice Coefficient
                if map_a.shape == map_b.shape:
                    dice_value = calculate_dice_coefficient(map_a, map_b)
                    dice_all_models_results.append({
                        'model': model_name,
                        'image_id': image_id,
                        'method_a': method_a_name,
                        'method_b': method_b_name,
                        'Dice_Coefficient': dice_value
                    })
                else:
                    print(f"⚠️ Warning: Saliency maps for {image_id} in model {model_name} have different shapes for {method_a_name} ({map_a.shape}) and {method_b_name} ({map_b.shape}). Skipping Dice Coefficient for this pair.")

    end_time_dice = time.time()
    total_calculation_time_dice = end_time_dice - start_time_dice

    print(f"Total Dice Coefficient calculation time: {total_calculation_time_dice:.2f} seconds")

    # Convert results to DataFrame
    df_dice_all_models = pd.DataFrame(dice_all_models_results)

    # Save to CSV in the same directory as other results
    output_csv_filename_dice = output_dir / 'dice_coefficient_all_models_full_matrix_results.csv'
    df_dice_all_models.to_csv(output_csv_filename_dice, index=False)

   # print(f"Dice Coefficient full matrix results for all models and method comparisons saved to {output_csv_filename_dice}")
   # print("First 5 rows of the Dice Coefficient results DataFrame:")
   # display(df_dice_all_models.head())

```markdown
## Soft IoU (Fuzzy Jaccard Index)

Soft IoU to miara podobieństwa między dwiema mapami saliency (lub dowolnymi mapami prawdopodobieństwa/ważności), która, w przeciwieństwie do tradycyjnego IoU, nie wymaga binarnej segmentacji. Zamiast tego, traktuje wartości pikseli jako "natężenie" lub "prawdopodobieństwo", pozwalając na bardziej subtelne porównanie.

Wzór na Soft IoU jest następujący:

$$ \text{Soft IoU}(P, G) = \frac{\sum_{i=1}^{N} \min(P_i, G_i)}{\sum_{i=1}^{N} \max(P_i, G_i)} $$

**Opis zmiennych:**

*   **$P$**: Pierwsza mapa saliency (lub mapa prawdopodobieństwa), którą porównujemy. Reprezentuje ją wektor spłaszczonych wartości pikseli $P = [P_1, P_2, ..., P_N]$, gdzie $P_i$ to wartość w  .

*   **$G$**: Druga mapa saliency (lub mapa prawdopodobieństwa), służąca jako punkt odniesienia dla porównania. Reprezentuje ją wektor spłaszczonych wartości pikseli $G = [G_1, G_2, ..., G_N]$, gdzie $G_i$ to wartość w i-tym pikselu.

*   **$N$**: Całkowita liczba pikseli w mapach $P$ i $G$ (zakładamy, że obie mapy mają te same wymiary).

*   **$\min(P_i, G_i)$**: Minimalna wartość spośród odpowiadających sobie pikseli $P_i$ i $G_i$. Suma tych wartości w liczniku reprezentuje "przecięcie" (intersection) map, uwzględniając ich natężenia.

*   **$\max(P_i, G_i)$**: Maksymalna wartość spośród odpowiadających sobie pikseli $P_i$ i $G_i$. Suma tych wartości w mianowniku reprezentuje "sumę" (union) map, uwzględniając ich natężenia.

**Kluczowe Kroki przed obliczeniem Soft IoU:**

Przed zastosowaniem powyższego wzoru, często konieczne jest **normalizowanie** map $P$ i $G$. Normalizacja zazwyczaj polega na przekształceniu wartości pikseli tak, aby stanowiły rozkład prawdopodobieństwa, tzn. sumowały się do 1. Może to wyglądać następująco:

1.  Upewnij się, że wszystkie wartości są nieujemne (np. przez `max(0, wartość)`).
2.  Dodaj małą stałą (epsilon) do każdej wartości, aby uniknąć dzielenia przez zero.
3.  Podziel każdą wartość piksela przez sumę wszystkich wartości w danej mapie.

To pozwala traktować mapy jako dystrybucje, co jest zgodne z intuicją Soft IoU jako miary podobieństwa rozkładów.
```

In [ ]:
import numpy as np
import cv2
from scipy.stats import pearsonr
from sklearn.metrics import auc, precision_recall_curve
from pathlib import Path
import torch
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import os
import time
import pandas as pd

# --- User-provided functions START ---

EPS = 1e-12

def normalize_to_probability_map(saliency):
    """Normalize a map so all pixel values sum to 1."""
    saliency = np.maximum(saliency - saliency.min(), 0)
    total = saliency.sum()
    return saliency / (total + EPS)


def pearson_correlation(map1, map2):
    """Pearson correlation between two flattened maps."""
    a, b = map1.flatten(), map2.flatten()
    if a.std() < EPS or b.std() < EPS:
        return 0.0
    return float(pearsonr(a, b)[0])

def similarity_score(map1, map2):
    """SIM metric (histogram intersection between normalized maps)."""
    a, b = normalize_to_probability_map(map1), normalize_to_probability_map(map2)
    return float(np.sum(np.minimum(a, b)))


def resize_to_match(image, target_shape):
    """Resize a saliency map to match image resolution."""
    # Ensure image is 2D if it's a saliency map
    if image.ndim > 2:
        image = image.squeeze() # Remove channel dimension if present
    return cv2.resize(image, (target_shape[1], target_shape[0]), interpolation=cv2.INTER_LINEAR)



def get_pixel_ranking_descending(saliency_map):
    """Return indices of pixels sorted from most to least important."""
    return np.argsort(-saliency_map.flatten())


def run_deletion_test(image, predict_fn, saliency, class_index, steps=20, replacement_value=0,
                       image_name=None, save_debug=False, debug_root="check_deletion"):
    """
    Remove most important pixels progressively and record class score.
    Lower AUC means a better explanation.
    """
    image = np.array(image, dtype=np.float32)
    image_to_plot = image.copy()
    # only for checking the masking is true - visualization
    if image.max() > 1.5:  # normalize if in [0,255]
        image_to_plot /= 255.0
    h, w = saliency.shape
    sorted_pixels = get_pixel_ranking_descending(saliency)
    total_pixels = h * w
    scores = []

    # Debug folder (if saving)
    if save_debug and image_name is not None:
        save_dir = Path(debug_root) / Path(image_name).stem
        save_dir.mkdir(parents=True, exist_ok=True)


    for step in range(steps + 1):
        num_remove = int(total_pixels * step / steps)
        mask = np.ones(total_pixels, bool)
        mask[sorted_pixels[:num_remove]] = False
        mask = mask.reshape(h, w)

        modified = image.copy()
        # Apply mask to all channels of the image
        if modified.ndim == 3: # If it's an RGB image
            for c in range(modified.shape[2]):
                modified[~mask, c] = replacement_value
        else: # Grayscale
            modified[~mask] = replacement_value

        modified_to_plot = image_to_plot.copy()
        if modified_to_plot.ndim == 3:
             for c in range(modified_to_plot.shape[2]):
                modified_to_plot[~mask, c] = replacement_value
        else:
            modified_to_plot[~mask] = replacement_value

        score = predict_fn(modified, class_index)
        scores.append(score)

        # Save every 10th step for inspection
        if save_debug and (step % 10 == 0) and image_name is not None:
            save_img = np.clip(modified_to_plot, 0, 1)
            save_img = (save_img * 255).astype(np.uint8)
            cv2.imwrite(str(save_dir / f"step_{step:03d}.png"), cv2.cvtColor(save_img, cv2.COLOR_RGB2BGR))


    x_axis = np.linspace(0, 1, steps + 1)
    return x_axis, np.array(scores)


def run_insertion_test(image, predict_fn, saliency, class_index, steps=20,
                       image_name=None, save_debug=False, debug_root="check_insertion"):
    """
    Start from a blurred baseline and insert important pixels gradually.
    Higher AUC means a better explanation.
    """
    image = np.array(image, dtype=np.float32)
    image_to_plot = image.copy()
    # only for checking the masking is true

    if image.max() > 1.5:  # normalize if in [0,255]
        image_to_plot /= 255.0
    h, w = saliency.shape
    sorted_pixels = get_pixel_ranking_descending(saliency)
    total_pixels = h * w
    blurred = cv2.GaussianBlur(image, (21, 21), 0)
    blurred_to_plot = cv2.GaussianBlur(image_to_plot, (21, 21), 0)
    scores = []

    # Debug folder (if saving)
    if save_debug and image_name is not None:
        save_dir = Path(debug_root) / Path(image_name).stem
        save_dir.mkdir(parents=True, exist_ok=True)

    for step in range(steps + 1):
        num_add = int(total_pixels * step / steps)
        mask = np.zeros(total_pixels, bool)
        mask[sorted_pixels[:num_add]] = True
        mask = mask.reshape(h, w)

        modified = blurred.copy()
        if modified.ndim == 3:
            for c in range(modified.shape[2]):
                modified[mask, c] = image[mask, c]
        else:
            modified[mask] = image[mask]

        modified_to_plot = blurred_to_plot.copy()
        if modified_to_plot.ndim == 3:
             for c in range(modified_to_plot.shape[2]):
                modified_to_plot[mask, c] = image_to_plot[mask, c]
        else:
            modified_to_plot[mask] = image_to_plot[mask]

        score = predict_fn(modified, class_index)
        scores.append(score)

        # Save every 10th step for inspection
        if save_debug and (step % 10 == 0) and image_name is not None:
            save_img = np.clip(modified_to_plot, 0, 1)
            save_img = (save_img * 255).astype(np.uint8)
            cv2.imwrite(str(save_dir / f"step_{step:03d}.png"), cv2.cvtColor(save_img, cv2.COLOR_RGB2BGR))


    x_axis = np.linspace(0, 1, steps + 1)
    return x_axis, np.array(scores)


def compute_auc(x_values, y_values):
    """Simple AUC helper."""
    # Ensure x_values are sorted for auc function
    sort_idx = np.argsort(x_values)
    x_values_sorted = x_values[sort_idx]
    y_values_sorted = y_values[sort_idx]
    return float(auc(x_values_sorted, y_values_sorted))

# --- User-provided functions END ---


# --- Placeholder Functions (ADAPT AS NEEDED) ---

def load_model_for_prediction(model_name: str, device: str):
    """
    Placeholder function to load a pre-trained model for prediction.
    Replace this with your actual model loading logic.
    """
    print(f"Loading placeholder model for {model_name}...")
    # Example for PyTorch: Replace with your actual model architecture and weights
    if model_name == 'efficientnet_b0':
        from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
        weights = EfficientNet_B0_Weights.DEFAULT
        model = efficientnet_b0(weights=weights).to(device)
        model.eval()
        return model, weights.transforms()
    elif model_name == 'resnet18':
        from torchvision.models import resnet18, ResNet18_Weights
        weights = ResNet18_Weights.DEFAULT
        model = resnet18(weights=weights).to(device)
        model.eval()
        return model, weights.transforms()
    elif model_name == 'mobilenet_v3_large':
        from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights
        weights = MobileNet_V3_Large_Weights.DEFAULT
        model = mobilenet_v3_large(weights=weights).to(device)
        model.eval()
        return model, weights.transforms()
    elif model_name == 'squeezenet':
        from torchvision.models import squeezenet1_0, SqueezeNet1_0_Weights
        weights = SqueezeNet1_0_Weights.DEFAULT
        model = squeezenet1_0(weights=weights).to(device)
        model.eval()
        return model, weights.transforms()
    else:
        raise NotImplementedError(f"Model {model_name} not supported in placeholder")

def get_image_path(image_id: str, base_search_dir: str = '/content/drive/MyDrive/XAI_Project/') -> str:
    """
    Determines the full path to an image based on its image_id by searching within
    a specified base directory and its subdirectories.
    The `image_id` is expected to be in the format 'cat_X' or 'dog_X'.
    """
    target_filename = f"{image_id}.jpg"
    search_root = Path(base_search_dir)

    # Use rglob to search for the file recursively.
    # rglob returns a generator, convert to list to check if any paths were found.
    found_paths = list(search_root.rglob(target_filename))

    if found_paths:
        # Return the first found path. If multiple exist, this might pick one arbitrarily.
        return str(found_paths[0])
    else:
        # If the file is not found, return an empty string.
        # The calling code already handles this by printing a warning.
        return ""


def model_predict_fn(image_np: np.ndarray, class_index: int, model, preprocessor, device: str) -> float:
    """
    Adapter for the predict_fn required by run_deletion_test/run_insertion_test.
    Takes a numpy array image (expected to be HWC, float32, in range [0, 255]),
    converts it to PIL, preprocesses, and gets prediction.
    """
    # Ensure image_np is in [0, 255] range and uint8 for PIL.Image.fromarray
    # This handles both cases: image_np already [0, 255] or image_np normalized [0, 1] then scaled
    if image_np.max() <= 1.0 + EPS and image_np.min() >= 0.0 - EPS: # If approximately normalized
        image_np_uint8 = np.clip(image_np * 255, 0, 255).astype(np.uint8)
    else: # Assume [0, 255] or similar, just clip
        image_np_uint8 = np.clip(image_np, 0, 255).astype(np.uint8)

    img_pil = Image.fromarray(image_np_uint8)

    input_tensor = preprocessor(img_pil).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(input_tensor)
        probabilities = F.softmax(output, dim=1)

    return probabilities[0, class_index].item()


print("Functions for AUC-based Insertion/Deletion tests loaded.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

results_auc_id = []

# Assuming all_loaded_data is available from previous cells
if 'all_loaded_data' not in locals() or not all_loaded_data:
    print("Error: `all_loaded_data` not found. Please run the data loading cell first.")
else:
    all_models_instances = {}
    total_start_time = time.time()

    # Ensure the output directory exists
    output_dir = Path('/content/drive/MyDrive/XAI_Project/results_multi_cam/results_all/')
    output_dir.mkdir(parents=True, exist_ok=True)

    for model_name, methods_data in all_loaded_data.items():
        print(f"\nProcessing model: {model_name}")
        model_instance = None
        preprocessor_instance = None

        try:
            # Load model and its preprocessor once per model_name
            # This makes sure the preprocessor is also consistent
            if model_name not in all_models_instances:
                model_instance, preprocessor_instance = load_model_for_prediction(model_name, device)
                all_models_instances[model_name] = (model_instance, preprocessor_instance)
            else:
                model_instance, preprocessor_instance = all_models_instances[model_name]

            # Define the predict_fn for this specific model and preprocessor
            current_predict_fn = lambda img_array, cls_idx: model_predict_fn(
                img_array, cls_idx, model_instance, preprocessor_instance, device
            )

        except NotImplementedError as e:
            print(f"Skipping model {model_name}: {e}")
            continue
        except Exception as e:
            print(f"Error loading model {model_name}: {e}. Skipping.")
            continue

        for method_name in methods_data.keys():
            print(f"  Processing method: {method_name}")

            for image_id, saliency_map in methods_data[method_name].items():
                image_path = get_image_path(image_id) # Get the path to the original image

                if not os.path.exists(image_path):
                    print(f"    Warning: Image file not found for {image_id} at {image_path}. Skipping.")
                    continue

                try:
                    # Load original image and convert to numpy [H, W, C], float32, range [0, 255]
                    original_img_pil = Image.open(image_path).convert('RGB')
                    original_img_np = np.array(original_img_pil, dtype=np.float32)

                    # Determine target class index (placeholder logic)
                    target_class_idx = 0 if 'cat' in image_id else 1 # Assuming cat=0, dog=1
                    # This might need to be adjusted if your model has different class mappings

                    # Resize saliency map to match original image dimensions
                    saliency_map_resized = resize_to_match(saliency_map, original_img_np.shape[:2])

                    # --- Insertion Test ---
                    ins_x, ins_y = run_insertion_test(
                        original_img_np, current_predict_fn, saliency_map_resized,
                        target_class_idx, steps=20, image_name=image_id, save_debug=False
                    )
                    ins_auc_score = compute_auc(ins_x, ins_y)
                    results_auc_id.append({
                        'model': model_name,
                        'method': method_name,
                        'image_id': image_id,
                        'test_type': 'insertion',
                        'AUC': ins_auc_score
                    })

                    # --- Deletion Test ---
                    del_x, del_y = run_deletion_test(
                        original_img_np, current_predict_fn, saliency_map_resized,
                        target_class_idx, steps=20, replacement_value=0, image_name=image_id, save_debug=False
                    )
                    del_auc_score = compute_auc(del_x, del_y)
                    results_auc_id.append({
                        'model': model_name,
                        'method': method_name,
                        'image_id': image_id,
                        'test_type': 'deletion',
                        'AUC': del_auc_score
                    })
                except Exception as e:
                    print(f"      Error processing {image_id} for {method_name}: {e}. Skipping.")

    total_end_time = time.time()
    print(f"\nTotal AUC Insertion/Deletion calculation time: {total_end_time - total_start_time:.2f} seconds")

    df_auc_id_results = pd.DataFrame(results_auc_id)
    output_csv_filename_auc_id = output_dir / 'auc_insertion_deletion_results_last.csv'
    df_auc_id_results.to_csv(output_csv_filename_auc_id, index=False)

    print(f"AUC Insertion/Deletion results saved to {output_csv_filename_auc_id}")
    print("First 5 rows of the results DataFrame:")
    display(df_auc_id_results.head())

In [ ]:
import itertools
import time
import pandas as pd
import numpy as np

# Function to calculate Intersection over Union (IoU)
def calculate_iou(map1: np.ndarray, map2: np.ndarray) -> float:
    # Ensure maps have the same shape
    if map1.shape != map2.shape:
        print(f"Warning: Maps have different shapes: {map1.shape} vs {map2.shape}. Returning 0.0")
        return 0.0

    # Binarize maps using their respective medians as thresholds
    # Handle constant arrays: if all values are the same, median is that value.
    # Using > median ensures there's at least some '1' if not all values are identical.

    if np.all(map1 == map1.flat[0]): # Check if map1 is constant
        bin_map1 = np.zeros_like(map1, dtype=bool)
    else:
        median_map1 = np.median(map1)
        bin_map1 = map1 > median_map1

    if np.all(map2 == map2.flat[0]): # Check if map2 is constant
        bin_map2 = np.zeros_like(map2, dtype=bool)
    else:
        median_map2 = np.median(map2)
        bin_map2 = map2 > median_map2

    # Calculate Intersection
    intersection = np.sum(np.logical_and(bin_map1, bin_map2))

    # Calculate Union
    union = np.sum(np.logical_or(bin_map1, bin_map2))

    if union == 0:
        return 0.0 # Avoid division by zero if both masks are empty
    else:
        return intersection / union

print("IoU calculation function defined.")

start_time_iou = time.time()

iou_all_models_results = []

# Iterate through each model
for model_name, methods_data in all_loaded_data.items():
    method_names = list(methods_data.keys())

    # Iterate through all pairs of methods, including self-comparison and reversed pairs
    for method_a_name, method_b_name in itertools.product(method_names, repeat=2):
        # Get the set of image IDs common to both methods for this model
        common_image_ids = set(methods_data[method_a_name].keys()).intersection(methods_data[method_b_name].keys())

        for image_id in common_image_ids:
            map_a = methods_data[method_a_name][image_id]
            map_b = methods_data[method_b_name][image_id]

            # Ensure maps have the same shape before calculating IoU
            if map_a.shape == map_b.shape:
                iou_value = calculate_iou(map_a, map_b)
                iou_all_models_results.append({
                    'model': model_name,
                    'image_id': image_id,
                    'method_a': method_a_name,
                    'method_b': method_b_name,
                    'IoU': iou_value
                })
            else:
                print(f"⚠️ Warning: Saliency maps for {image_id} in model {model_name} have different shapes for {method_a_name} ({map_a.shape}) and {method_b_name} ({map_b.shape}). Skipping IoU for this pair.")

end_time_iou = time.time()
total_calculation_time_iou = end_time_iou - start_time_iou

print(f"Total IoU calculation time: {total_calculation_time_iou:.2f} seconds")

# Convert results to DataFrame
df_iou_all_models = pd.DataFrame(iou_all_models_results)

# Save to CSV in the same directory as other results
output_csv_filename_iou = '/content/drive/MyDrive/XAI_Project/results_multi_cam/results_all/iou_all_models_full_matrix_results.csv'
df_iou_all_models.to_csv(output_csv_filename_iou, index=False)

print(f"IoU full matrix results for all models and method comparisons saved to {output_csv_filename_iou}")
print("First 5 rows of the IoU results DataFrame:")
display(df_iou_all_models.head())

In [ ]:
import numpy as np
import pandas as pd
import time
import itertools

def calculate_soft_iou(heatmap1, heatmap2):
    """
    Oblicza Soft IoU (Fuzzy Jaccard Index) między dwiema heatmapami.
    Heatmapy powinny być znormalizowane do zakresu [0, 1] i mieć te same wymiary.
    """
    # Upewnienie się, że wejścia to tablice numpy
    h1 = np.array(heatmap1, dtype=np.float32)
    h2 = np.array(heatmap2, dtype=np.float32)

    # Dodanie epsilon, aby uniknąć problemów z dzieleniem przez zero
    # i aby zapewnić, że wartości są nieujemne dla normalizacji.
    epsilon = 1e-10
    h1 = np.maximum(h1, 0) + epsilon
    h2 = np.maximum(h2, 0) + epsilon

    # Znormalizowanie heatmap do zakresu [0, 1] proporcjonalnie do ich sumy
    # To ważne dla Soft IoU, które często zakłada, że 'heat' sumuje się do 1.
    h1 = h1 / h1.sum()
    h2 = h2 / h2.sum()

    # Przecięcie (Intersection) to suma minimów w każdym punkcie
    intersection = np.sum(np.minimum(h1, h2))

    # Suma (Union) to suma maksimów w każdym punkcie
    union = np.sum(np.maximum(h1, h2))

    # Obliczenie IoU
    iou = intersection / union if union != 0 else 0.0
    return iou

print("Soft IoU (Fuzzy Jaccard Index) function defined.")

start_time_soft_iou = time.time()

soft_iou_all_models_results = []

# Assuming all_loaded_data is available from previous cells
if 'all_loaded_data' not in locals() or not all_loaded_data:
    print("Error: `all_loaded_data` not found. Please run the data loading cell first.")
else:
    # Iterate through each model
    for model_name, methods_data in all_loaded_data.items():
        method_names = list(methods_data.keys())

        # Iterate through all pairs of methods, including self-comparison and reversed pairs
        for method_a_name, method_b_name in itertools.product(method_names, repeat=2):
            # Get the set of image IDs common to both methods for this model
            common_image_ids = set(methods_data[method_a_name].keys()).intersection(methods_data[method_b_name].keys())

            for image_id in common_image_ids:
                map_a = methods_data[method_a_name][image_id]
                map_b = methods_data[method_b_name][image_id]

                # Ensure maps have the same shape before calculating Soft IoU
                if map_a.shape == map_b.shape:
                    soft_iou_value = calculate_soft_iou(map_a, map_b)
                    soft_iou_all_models_results.append({
                        'model': model_name,
                        'image_id': image_id,
                        'method_a': method_a_name,
                        'method_b': method_b_name,
                        'Soft_IoU': soft_iou_value
                    })
                else:
                    print(f"⚠️ Warning: Saliency maps for {image_id} in model {model_name} have different shapes for {method_a_name} ({map_a.shape}) and {method_b_name} ({map_b.shape}). Skipping Soft IoU for this pair.")

    end_time_soft_iou = time.time()
    total_calculation_time_soft_iou = end_time_soft_iou - start_time_soft_iou

    print(f"Total Soft IoU calculation time: {total_calculation_time_soft_iou:.2f} seconds")

    # Convert results to DataFrame
    df_soft_iou_all_models = pd.DataFrame(soft_iou_all_models_results)

    # Save to CSV in the same directory as other results
    output_csv_filename_soft_iou = '/content/drive/MyDrive/XAI_Project/results_multi_cam/results_all/soft_iou_all_models_full_matrix_results.csv'
    df_soft_iou_all_models.to_csv(output_csv_filename_soft_iou, index=False)

    print(f"Soft IoU full matrix results for all models and method comparisons saved to {output_csv_filename_soft_iou}")
    print("First 5 rows of the Soft IoU results DataFrame:")
    display(df_soft_iou_all_models.head())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the results
iou_df = pd.read_csv('/content/drive/MyDrive/XAI_Project/results_multi_cam/results_all/iou_all_models_full_matrix_results.csv')
soft_iou_df = pd.read_csv('/content/drive/MyDrive/XAI_Project/results_multi_cam/results_all/soft_iou_all_models_full_matrix_results.csv')

# Filter out self-comparisons (where method_a == method_b) as they always result in 1.0
iou_df_filtered = iou_df[iou_df['method_a'] != iou_df['method_b']].copy()
soft_iou_df_filtered = soft_iou_df[soft_iou_df['method_a'] != soft_iou_df['method_b']].copy()

# Merge dataframes for easier plotting (optional, but good for consistent visualization)
# Rename columns for clarity before merging
iou_df_filtered['metric_type'] = 'IoU'
iou_df_filtered.rename(columns={'IoU': 'value'}, inplace=True)

soft_iou_df_filtered['metric_type'] = 'Soft IoU'
soft_iou_df_filtered.rename(columns={'Soft_IoU': 'value'}, inplace=True)

# Concatenate for plotting
combined_df = pd.concat([iou_df_filtered[['model', 'image_id', 'method_a', 'method_b', 'metric_type', 'value']],
                         soft_iou_df_filtered[['model', 'image_id', 'method_a', 'method_b', 'metric_type', 'value']]],
                        ignore_index=True)

# Create the visualization
plt.figure(figsize=(14, 8))
sns.violinplot(x='metric_type', y='value', hue='metric_type', data=combined_df, palette={'IoU': 'skyblue', 'Soft IoU': 'lightcoral'})
plt.title('Distribution Comparison of IoU and Soft IoU (Excluding Self-Comparisons)')
plt.xlabel('Metric Type')
plt.ylabel('Value')
plt.ylim(-0.05, 1.05) # Ensure full range for correlation metrics
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
import itertools
import time
import pandas as pd
import numpy as np

# Ensure calculate_iou is defined (re-define or assume it's already in kernel)
# Copying it here for self-containment of the new cell
def calculate_iou(map1: np.ndarray, map2: np.ndarray) -> float:
    # Ensure maps have the same shape
    if map1.shape != map2.shape:
        print(f"Warning: Maps have different shapes: {map1.shape} vs {map2.shape}. Returning 0.0")
        return 0.0

    if np.all(map1 == map1.flat[0]): # Check if map1 is constant
        bin_map1 = np.zeros_like(map1, dtype=bool)
    else:
        median_map1 = np.median(map1)
        bin_map1 = map1 > median_map1

    if np.all(map2 == map2.flat[0]): # Check if map2 is constant
        bin_map2 = np.zeros_like(map2, dtype=bool)
    else:
        median_map2 = np.median(map2)
        bin_map2 = map2 > median_map2

    intersection = np.sum(np.logical_and(bin_map1, bin_map2))
    union = np.sum(np.logical_or(bin_map1, bin_map2))

    if union == 0:
        return 0.0
    else:
        return intersection / union

print("IoU calculation function (re-)defined for this scope.")

start_time_iou = time.time()

iou_specific_results = []

# Iterate through the specific_saliency_data
if 'specific_saliency_data' in locals() and specific_saliency_data:
    for model_name, methods_data in specific_saliency_data.items():
        method_names = list(methods_data.keys())

        for method_a_name, method_b_name in itertools.product(method_names, repeat=2):
            common_image_ids = set(methods_data[method_a_name].keys()).intersection(methods_data[method_b_name].keys())

            for image_id in common_image_ids:
                map_a = methods_data[method_a_name][image_id]
                map_b = methods_data[method_b_name][image_id]

                if map_a.shape == map_b.shape:
                    iou_value = calculate_iou(map_a, map_b)
                    iou_specific_results.append({
                        'model': model_name,
                        'image_id': image_id,
                        'method_a': method_a_name,
                        'method_b': method_b_name,
                        'IoU': iou_value
                    })
                else:
                    print(f"⚠️ Warning: Saliency maps for {image_id} in model {model_name} have different shapes for {method_a_name} ({map_a.shape}) and {method_b_name} ({map_b.shape}). Skipping IoU for this pair.")
else:
    print("Error: `specific_saliency_data` not found or empty. Please ensure the specific data loading cell was run.")

end_time_iou = time.time()
total_calculation_time_iou = end_time_iou - start_time_iou

print(f"Total IoU calculation time for specific data: {total_calculation_time_iou:.2f} seconds")

df_iou_specific = pd.DataFrame(iou_specific_results)
output_csv_filename_iou_specific = '/content/drive/MyDrive/XAI_Project/results_multi_cam/results_all/iou_specific_saliency_maps.csv'
df_iou_specific.to_csv(output_csv_filename_iou_specific, index=False)

print(f"IoU results for specific saliency maps saved to {output_csv_filename_iou_specific}")
print("First 5 rows of the specific IoU results DataFrame:")
display(df_iou_specific.head())

In [ ]:
import numpy as np
from pathlib import Path
import re # Added for regex

def load_specific_saliency_maps(file_paths):
    loaded_data = {}
    # Use a single, consistent model_name for these specific comparisons
    # so that methods can be compared against each other for the same image.
    comparison_group_model_name = "specific_comparison_model"

    if comparison_group_model_name not in loaded_data:
        loaded_data[comparison_group_model_name] = {}

    for path_str in file_paths:
        path = Path(path_str)
        try:
            saliency_map = np.load(path_str)
            # Ensure saliency map is 2D by squeezing any singleton dimensions
            saliency_map = saliency_map.squeeze()

            image_id_match = re.match(r'(cat_\d+|dog_\d+)', path.stem)
            image_id = image_id_match.group(0) if image_id_match else path.stem # Fallback if no cat/dog ID

            method_name = "unknown_method" # Default
            # Attempt to extract method name based on substrings in the filename
            if "FullGrad" in path.stem:
                method_name = "FullGrad"
            elif "GradCAM" in path.stem:
                method_name = "GradCAM"
            elif "integrated_saliency" in path.stem:
                method_name = "integrated_saliency"
            elif "LIME" in path.stem:
                method_name = "LIME"
            # Add other specific method names as needed

            if method_name not in loaded_data[comparison_group_model_name]:
                loaded_data[comparison_group_model_name][method_name] = {}
            loaded_data[comparison_group_model_name][method_name][image_id] = saliency_map
            print(f"Loaded {path.name} as model='{comparison_group_model_name}', method='{method_name}', image_id='{image_id}'")
        except Exception as e:
            print(f"⚠️ Error loading {path_str}: {e}")
    return loaded_data

file_paths_to_load = [
    '/content/cat_1_efficientnet_b0_Egyptian_cat_FullGrad_raw.npy',
    '/content/cat_1_integrated_saliency.npy'
]

specific_saliency_data = load_specific_saliency_maps(file_paths_to_load)

print("\nStructure of loaded specific saliency data:")
for model, methods in specific_saliency_data.items():
    print(f"  Model: {model}")
    for method, images in methods.items():
        print(f"    Method: {method} (Images: {len(images)})")
        for img_id, saliency_map in images.items():
            print(f"      Image ID: {img_id}, Shape: {saliency_map.shape}, Dtype: {saliency_map.dtype}")

In [ ]:
import itertools
import time
import pandas as pd
import numpy as np
from numba import jit, cuda # Import cuda for GPU optimization
from scipy.stats import pearsonr

# CUDA kernel for element-wise calculations (numerator parts, sum of squared deviations parts)
@cuda.jit
def _pcc_parts_kernel(arr1_d, arr2_d, mean1, mean2, numerator_parts_d, dev1_parts_d, dev2_parts_d):
    idx = cuda.grid(1)
    if idx < arr1_d.size:
        val1 = arr1_d[idx] - mean1
        val2 = arr2_d[idx] - mean2
        numerator_parts_d[idx] = val1 * val2
        dev1_parts_d[idx] = val1 * val1
        dev2_parts_d[idx] = val2 * val2

# Define GPU-optimized calculate_pcc function
def calculate_pcc_gpu(arr1: np.ndarray, arr2: np.ndarray) -> float:
    # Ensure data is float32 for CUDA compatibility
    flat_arr1 = arr1.flatten().astype(np.float32)
    flat_arr2 = arr2.flatten().astype(np.float32)

    if len(flat_arr1) != len(flat_arr2):
        return 0.0

    # Calculate means on CPU. For very large arrays, this could also be parallelized on GPU.
    mean1 = np.mean(flat_arr1)
    mean2 = np.mean(flat_arr2)

    # Transfer data to device (GPU)
    arr1_d = cuda.to_device(flat_arr1)
    arr2_d = cuda.to_device(flat_arr2)

    # Allocate device arrays for intermediate results
    numerator_parts_d = cuda.device_array(flat_arr1.shape, dtype=np.float32)
    dev1_parts_d = cuda.device_array(flat_arr1.shape, dtype=np.float32)
    dev2_parts_d = cuda.device_array(flat_arr1.shape, dtype=np.float32)

    # Configure and launch kernel for element-wise operations
    threadsperblock = 256
    blockspergrid = (flat_arr1.size + (threadsperblock - 1)) // threadsperblock
    _pcc_parts_kernel[blockspergrid, threadsperblock](arr1_d, arr2_d, mean1, mean2,
                                                      numerator_parts_d, dev1_parts_d, dev2_parts_d)
    cuda.synchronize() # Wait for the kernel to complete

    # Copy intermediate results back to host for sum reduction.
    # For ultimate GPU performance, these sums could also be performed on the GPU using
    # `numba.cuda.reduce` or custom reduction kernels.
    numerator = numerator_parts_d.copy_to_host().sum()
    sum_sq_dev1 = dev1_parts_d.copy_to_host().sum()
    sum_sq_dev2 = dev2_parts_d.copy_to_host().sum()

    denominator = np.sqrt(sum_sq_dev1) * np.sqrt(sum_sq_dev2)

    if denominator == 0:
        return 0.0
    else:
        pcc = numerator / denominator
        if np.isnan(pcc):
            return 0.0
        return pcc

# Keep the original CPU-optimized version as a fallback or for explicit comparison
@jit(nopython=True)
def calculate_pcc_cpu(arr1: np.ndarray, arr2: np.ndarray) -> float:
    # 3. Flatten both input arrays into 1D vectors
    flat_arr1 = arr1.flatten()
    flat_arr2 = arr2.flatten()

    # Ensure both arrays have the same length after flattening
    if len(flat_arr1) != len(flat_arr2):
        return 0.0 # Or raise an error, depending on desired behavior

    # Calculate means
    mean1 = np.mean(flat_arr1)
    mean2 = np.mean(flat_arr2)

    # Calculate numerator (sum of (x_i - mean_x)(y_i - mean_y))
    numerator = np.sum((flat_arr1 - mean1) * (flat_arr2 - mean2))

    # Calculate denominators (sum of (x_i - mean_x)^2 and (y_i - mean_y)^2)
    sum_sq_dev1 = np.sum((flat_arr1 - mean1)**2)
    sum_sq_dev2 = np.sum((flat_arr2 - mean2)**2)

    # Handle potential division by zero (e.g., if one array has zero variance)
    denominator = np.sqrt(sum_sq_dev1) * np.sqrt(sum_sq_dev2)

    if denominator == 0:
        return 0.0 # If there's no variance in one or both, PCC is undefined, return 0
    else:
        pcc = numerator / denominator
        # Handle potential NaN if inputs contain NaNs and lead to NaNs in calculation
        if np.isnan(pcc):
            return 0.0
        return pcc

# Decide which version to use based on CUDA availability
try:
    if cuda.is_available():
        calculate_pcc = calculate_pcc_gpu
        print("Using GPU-optimized `calculate_pcc`.")
    else:
        calculate_pcc = calculate_pcc_cpu
        print("CUDA not available. Falling back to CPU-optimized `calculate_pcc`.")
except Exception as e:
    print(f"Error checking CUDA availability: {e}. Falling back to CPU-optimized `calculate_pcc`.")
    calculate_pcc = calculate_pcc_cpu

print("Libraries imported and `calculate_pcc` function defined and optimized with Numba (GPU or CPU fallback).")

start_time_pcc = time.time()

pcc_specific_results = []

# Iterate through the specific_saliency_data
if 'specific_saliency_data' in locals() and specific_saliency_data:
    for model_name, methods_data in specific_saliency_data.items():
        method_names = list(methods_data.keys())

        for method_a_name, method_b_name in itertools.product(method_names, repeat=2):
            common_image_ids = set(methods_data[method_a_name].keys()).intersection(methods_data[method_b_name].keys())

            for image_id in common_image_ids:
                map_a = methods_data[method_a_name][image_id]
                map_b = methods_data[method_b_name][image_id]

                if map_a.shape == map_b.shape:
                    pcc_value = calculate_pcc(map_a, map_b)
                    pcc_specific_results.append({
                        'model': model_name,
                        'image_id': image_id,
                        'method_a': method_a_name,
                        'method_b': method_b_name,
                        'PCC': pcc_value
                    })
                else:
                    print(f"⚠️ Warning: Saliency maps for {image_id} in model {model_name} have different shapes for {method_a_name} ({map_a.shape}) and {method_b_name} ({map_b.shape}). Skipping PCC for this pair.")
else:
    print("Error: `specific_saliency_data` not found or empty. Please ensure the specific data loading cell was run.")

end_time_pcc = time.time()
total_calculation_time_pcc = end_time_pcc - start_time_pcc

print(f"Total PCC calculation time for specific data: {total_calculation_time_pcc:.2f} seconds")

df_pcc_specific = pd.DataFrame(pcc_specific_results)
output_csv_filename_pcc_specific = '/content/drive/MyDrive/XAI_Project/results_multi_cam/results_all/pcc_specific_saliency_maps.csv'
df_pcc_specific.to_csv(output_csv_filename_pcc_specific, index=False)

print(f"PCC results for specific saliency maps saved to {output_csv_filename_pcc_specific}")
print("First 5 rows of the specific PCC results DataFrame:")
display(df_pcc_specific.head())

In [ ]:
import itertools
import time
import pandas as pd
import numpy as np

# Ensure calculate_iou is defined (re-define or assume it's already in kernel)
# Copying it here for self-containment of the new cell
def calculate_iou(map1: np.ndarray, map2: np.ndarray) -> float:
    # Ensure maps have the same shape
    if map1.shape != map2.shape:
        print(f"Warning: Maps have different shapes: {map1.shape} vs {map2.shape}. Returning 0.0")
        return 0.0

    if np.all(map1 == map1.flat[0]): # Check if map1 is constant
        bin_map1 = np.zeros_like(map1, dtype=bool)
    else:
        median_map1 = np.median(map1)
        bin_map1 = map1 > median_map1

    if np.all(map2 == map2.flat[0]): # Check if map2 is constant
        bin_map2 = np.zeros_like(map2, dtype=bool)
    else:
        median_map2 = np.median(map2)
        bin_map2 = map2 > median_map2

    intersection = np.sum(np.logical_and(bin_map1, bin_map2))
    union = np.sum(np.logical_or(bin_map1, bin_map2))

    if union == 0:
        return 0.0
    else:
        return intersection / union

print("IoU calculation function (re-)defined for this scope.")

start_time_iou = time.time()

iou_specific_results = []

# Iterate through the specific_saliency_data
if 'specific_saliency_data' in locals() and specific_saliency_data:
    for model_name, methods_data in specific_saliency_data.items():
        method_names = list(methods_data.keys())

        for method_a_name, method_b_name in itertools.product(method_names, repeat=2):
            common_image_ids = set(methods_data[method_a_name].keys()).intersection(methods_data[method_b_name].keys())

            for image_id in common_image_ids:
                map_a = methods_data[method_a_name][image_id]
                map_b = methods_data[method_b_name][image_id]

                if map_a.shape == map_b.shape:
                    iou_value = calculate_iou(map_a, map_b)
                    iou_specific_results.append({
                        'model': model_name,
                        'image_id': image_id,
                        'method_a': method_a_name,
                        'method_b': method_b_name,
                        'IoU': iou_value
                    })
                else:
                    print(f"⚠️ Warning: Saliency maps for {image_id} in model {model_name} have different shapes for {method_a_name} ({map_a.shape}) and {method_b_name} ({map_b.shape}). Skipping IoU for this pair.")
else:
    print("Error: `specific_saliency_data` not found or empty. Please ensure the specific data loading cell was run.")

end_time_iou = time.time()
total_calculation_time_iou = end_time_iou - start_time_iou

print(f"Total IoU calculation time for specific data: {total_calculation_time_iou:.2f} seconds")

df_iou_specific = pd.DataFrame(iou_specific_results)
output_csv_filename_iou_specific = '/content/drive/MyDrive/XAI_Project/results_multi_cam/results_all/iou_specific_saliency_maps.csv'
df_iou_specific.to_csv(output_csv_filename_iou_specific, index=False)

print(f"IoU results for specific saliency maps saved to {output_csv_filename_iou_specific}")
print("First 5 rows of the specific IoU results DataFrame:")
display(df_iou_specific.head())

In [ ]:
import itertools
import time
import pandas as pd
import numpy as np

# Ensure calculate_soft_iou is defined (re-define or assume it's already in kernel)
# Copying it here for self-containment of the new cell
def calculate_soft_iou(heatmap1, heatmap2):
    h1 = np.array(heatmap1, dtype=np.float32)
    h2 = np.array(heatmap2, dtype=np.float32)

    epsilon = 1e-10
    h1 = np.maximum(h1, 0) + epsilon
    h2 = np.maximum(h2, 0) + epsilon

    h1 = h1 / h1.sum()
    h2 = h2 / h2.sum()

    intersection = np.sum(np.minimum(h1, h2))
    union = np.sum(np.maximum(h1, h2))

    iou = intersection / union if union != 0 else 0.0
    return iou

print("Soft IoU (Fuzzy Jaccard Index) function (re-)defined for this scope.")

start_time_soft_iou = time.time()

soft_iou_specific_results = []

# Iterate through the specific_saliency_data
if 'specific_saliency_data' in locals() and specific_saliency_data:
    for model_name, methods_data in specific_saliency_data.items():
        method_names = list(methods_data.keys())

        for method_a_name, method_b_name in itertools.product(method_names, repeat=2):
            common_image_ids = set(methods_data[method_a_name].keys()).intersection(methods_data[method_b_name].keys())

            for image_id in common_image_ids:
                map_a = methods_data[method_a_name][image_id]
                map_b = methods_data[method_b_name][image_id]

                if map_a.shape == map_b.shape:
                    soft_iou_value = calculate_soft_iou(map_a, map_b)
                    soft_iou_specific_results.append({
                        'model': model_name,
                        'image_id': image_id,
                        'method_a': method_a_name,
                        'method_b': method_b_name,
                        'Soft_IoU': soft_iou_value
                    })
                else:
                    print(f"⚠️ Warning: Saliency maps for {image_id} in model {model_name} have different shapes for {method_a_name} ({map_a.shape}) and {method_b_name} ({map_b.shape}). Skipping Soft IoU for this pair.")
else:
    print("Error: `specific_saliency_data` not found or empty. Please ensure the specific data loading cell was run.")

end_time_soft_iou = time.time()
total_calculation_time_soft_iou = end_time_soft_iou - start_time_soft_iou

print(f"Total Soft IoU calculation time for specific data: {total_calculation_time_soft_iou:.2f} seconds")

df_soft_iou_specific = pd.DataFrame(soft_iou_specific_results)
output_csv_filename_soft_iou_specific = '/content/drive/MyDrive/XAI_Project/results_multi_cam/results_all/soft_iou_specific_saliency_maps.csv'
df_soft_iou_specific.to_csv(output_csv_filename_soft_iou_specific, index=False)

print(f"Soft IoU results for specific saliency maps saved to {output_csv_filename_soft_iou_specific}")
print("First 5 rows of the specific Soft IoU results DataFrame:")
display(df_soft_iou_specific.head())

In [ ]:
import itertools
import time
import pandas as pd
import numpy as np

# Ensure calculate_soft_iou is defined (re-define or assume it's already in kernel)
# Copying it here for self-containment of the new cell
def calculate_soft_iou(heatmap1, heatmap2):
    h1 = np.array(heatmap1, dtype=np.float32)
    h2 = np.array(heatmap2, dtype=np.float32)

    epsilon = 1e-10
    h1 = np.maximum(h1, 0) + epsilon
    h2 = np.maximum(h2, 0) + epsilon

    h1 = h1 / h1.sum()
    h2 = h2 / h2.sum()

    intersection = np.sum(np.minimum(h1, h2))
    union = np.sum(np.maximum(h1, h2))

    iou = intersection / union if union != 0 else 0.0
    return iou

print("Soft IoU (Fuzzy Jaccard Index) function (re-)defined for this scope.")

start_time_soft_iou = time.time()

soft_iou_specific_results = []

# Iterate through the specific_saliency_data
if 'specific_saliency_data' in locals() and specific_saliency_data:
    for model_name, methods_data in specific_saliency_data.items():
        method_names = list(methods_data.keys())

        for method_a_name, method_b_name in itertools.product(method_names, repeat=2):
            common_image_ids = set(methods_data[method_a_name].keys()).intersection(methods_data[method_b_name].keys())

            for image_id in common_image_ids:
                map_a = methods_data[method_a_name][image_id]
                map_b = methods_data[method_b_name][image_id]

                if map_a.shape == map_b.shape:
                    soft_iou_value = calculate_soft_iou(map_a, map_b)
                    soft_iou_specific_results.append({
                        'model': model_name,
                        'image_id': image_id,
                        'method_a': method_a_name,
                        'method_b': method_b_name,
                        'Soft_IoU': soft_iou_value
                    })
                else:
                    print(f"⚠️ Warning: Saliency maps for {image_id} in model {model_name} have different shapes for {method_a_name} ({map_a.shape}) and {method_b_name} ({map_b.shape}). Skipping Soft IoU for this pair.")
else:
    print("Error: `specific_saliency_data` not found or empty. Please ensure the specific data loading cell was run.")

end_time_soft_iou = time.time()
total_calculation_time_soft_iou = end_time_soft_iou - start_time_soft_iou

print(f"Total Soft IoU calculation time for specific data: {total_calculation_time_soft_iou:.2f} seconds")

df_soft_iou_specific = pd.DataFrame(soft_iou_specific_results)
output_csv_filename_soft_iou_specific = '/content/drive/MyDrive/XAI_Project/results_multi_cam/results_all/soft_iou_specific_saliency_maps.csv'
df_soft_iou_specific.to_csv(output_csv_filename_soft_iou_specific, index=False)

print(f"Soft IoU results for specific saliency maps saved to {output_csv_filename_soft_iou_specific}")
print("First 5 rows of the specific Soft IoU results DataFrame:")
display(df_soft_iou_specific.head())

In [ ]:
import itertools
import time
import pandas as pd
import numpy as np

# Function to calculate Intersection over Union (IoU)
def calculate_iou(map1: np.ndarray, map2: np.ndarray) -> float:
    # Ensure maps have the same shape
    if map1.shape != map2.shape:
        print(f"Warning: Maps have different shapes: {map1.shape} vs {map2.shape}. Returning 0.0")
        return 0.0

    # Binarize maps using their respective medians as thresholds
    # Handle constant arrays: if all values are the same, median is that value.
    # Using > median ensures there's at least some '1' if not all values are identical.

    if np.all(map1 == map1.flat[0]): # Check if map1 is constant
        bin_map1 = np.zeros_like(map1, dtype=bool)
    else:
        median_map1 = np.median(map1)
        bin_map1 = map1 > median_map1

    if np.all(map2 == map2.flat[0]): # Check if map2 is constant
        bin_map2 = np.zeros_like(map2, dtype=bool)
    else:
        median_map2 = np.median(map2)
        bin_map2 = map2 > median_map2

    # Calculate Intersection
    intersection = np.sum(np.logical_and(bin_map1, bin_map2))

    # Calculate Union
    union = np.sum(np.logical_or(bin_map1, bin_map2))

    if union == 0:
        return 0.0 # Avoid division by zero if both masks are empty
    else:
        return intersection / union

print("IoU calculation function defined.")

start_time_iou = time.time()

iou_all_models_results = []

# Iterate through each model
for model_name, methods_data in all_loaded_data.items():
    method_names = list(methods_data.keys())

    # Iterate through all pairs of methods, including self-comparison and reversed pairs
    for method_a_name, method_b_name in itertools.product(method_names, repeat=2):
        # Get the set of image IDs common to both methods for this model
        common_image_ids = set(methods_data[method_a_name].keys()).intersection(methods_data[method_b_name].keys())

        for image_id in common_image_ids:
            map_a = methods_data[method_a_name][image_id]
            map_b = methods_data[method_b_name][image_id]

            # Ensure maps have the same shape before calculating IoU
            if map_a.shape == map_b.shape:
                iou_value = calculate_iou(map_a, map_b)
                iou_all_models_results.append({
                    'model': model_name,
                    'image_id': image_id,
                    'method_a': method_a_name,
                    'method_b': method_b_name,
                    'IoU': iou_value
                })
            else:
                print(f"⚠️ Warning: Saliency maps for {image_id} in model {model_name} have different shapes for {method_a_name} ({map_a.shape}) and {method_b_name} ({map_b.shape}). Skipping IoU for this pair.")

end_time_iou = time.time()
total_calculation_time_iou = end_time_iou - start_time_iou

print(f"Total IoU calculation time: {total_calculation_time_iou:.2f} seconds")

# Convert results to DataFrame
df_iou_all_models = pd.DataFrame(iou_all_models_results)

# Save to CSV in the same directory as other results
output_csv_filename_iou = '/content/drive/MyDrive/XAI_Project/results_multi_cam/results_all/iou_all_models_full_matrix_results.csv'
df_iou_all_models.to_csv(output_csv_filename_iou, index=False)

print(f"IoU full matrix results for all models and method comparisons saved to {output_csv_filename_iou}")
print("First 5 rows of the IoU results DataFrame:")
display(df_iou_all_models.head())

In [ ]:
if 'all_loaded_data' in locals() and all_loaded_data:
    print("Załadowane modele:")
    for model_name in all_loaded_data.keys():
        print(f"- {model_name}")
elif 'all_loaded_data' in locals() and not all_loaded_data:
    print("Zmienna 'all_loaded_data' jest pusta. Nie załadowano żadnych modeli.")
else:
    print("Zmienna 'all_loaded_data' nie została jeszcze zdefiniowana. Upewnij się, że komórka ładująca dane została uruchomiona.")

In [ ]:
import itertools
import time
import pandas as pd
import numpy as np
from numba import jit
from scipy.stats import pearsonr # For comparison/fallback, though Numba implementation will be used

# 2. Define calculate_pcc function with Numba optimization
@jit(nopython=True)
def calculate_pcc(arr1: np.ndarray, arr2: np.ndarray) -> float:
    # 3. Flatten both input arrays into 1D vectors
    flat_arr1 = arr1.flatten()
    flat_arr2 = arr2.flatten()

    # Ensure both arrays have the same length after flattening
    if len(flat_arr1) != len(flat_arr2):
        return 0.0 # Or raise an error, depending on desired behavior

    # Calculate means
    mean1 = np.mean(flat_arr1)
    mean2 = np.mean(flat_arr2)

    # Calculate numerator (sum of (x_i - mean_x)(y_i - mean_y))
    numerator = np.sum((flat_arr1 - mean1) * (flat_arr2 - mean2))

    # Calculate denominators (sum of (x_i - mean_x)^2 and (y_i - mean_y)^2)
    sum_sq_dev1 = np.sum((flat_arr1 - mean1)**2)
    sum_sq_dev2 = np.sum((flat_arr2 - mean2)**2)

    # Handle potential division by zero (e.g., if one array has zero variance)
    denominator = np.sqrt(sum_sq_dev1) * np.sqrt(sum_sq_dev2)

    if denominator == 0:
        return 0.0 # If there's no variance in one or both, PCC is undefined, return 0
    else:
        pcc = numerator / denominator
        # Handle potential NaN if inputs contain NaNs and lead to NaNs in calculation
        if np.isnan(pcc):
            return 0.0
        return pcc

print("Libraries imported and `calculate_pcc` function defined and optimized with Numba.")

In [ ]:
import itertools
import time
import pandas as pd
import numpy as np
from numba import jit, cuda # Import cuda for GPU optimization
from scipy.stats import pearsonr

# CUDA kernel for element-wise calculations (numerator parts, sum of squared deviations parts)
@cuda.jit
def _pcc_parts_kernel(arr1_d, arr2_d, mean1, mean2, numerator_parts_d, dev1_parts_d, dev2_parts_d):
    idx = cuda.grid(1)
    if idx < arr1_d.size:
        val1 = arr1_d[idx] - mean1
        val2 = arr2_d[idx] - mean2
        numerator_parts_d[idx] = val1 * val2
        dev1_parts_d[idx] = val1 * val1
        dev2_parts_d[idx] = val2 * val2

# Define GPU-optimized calculate_pcc function
def calculate_pcc_gpu(arr1: np.ndarray, arr2: np.ndarray) -> float:
    # Ensure data is float32 for CUDA compatibility
    flat_arr1 = arr1.flatten().astype(np.float32)
    flat_arr2 = arr2.flatten().astype(np.float32)

    if len(flat_arr1) != len(flat_arr2):
        return 0.0

    # Calculate means on CPU. For very large arrays, this could also be parallelized on GPU.
    mean1 = np.mean(flat_arr1)
    mean2 = np.mean(flat_arr2)

    # Transfer data to device (GPU)
    arr1_d = cuda.to_device(flat_arr1)
    arr2_d = cuda.to_device(flat_arr2)

    # Allocate device arrays for intermediate results
    numerator_parts_d = cuda.device_array(flat_arr1.shape, dtype=np.float32)
    dev1_parts_d = cuda.device_array(flat_arr1.shape, dtype=np.float32)
    dev2_parts_d = cuda.device_array(flat_arr1.shape, dtype=np.float32)

    # Configure and launch kernel for element-wise operations
    threadsperblock = 256
    blockspergrid = (flat_arr1.size + (threadsperblock - 1)) // threadsperblock
    _pcc_parts_kernel[blockspergrid, threadsperblock](arr1_d, arr2_d, mean1, mean2,
                                                      numerator_parts_d, dev1_parts_d, dev2_parts_d)
    cuda.synchronize() # Wait for the kernel to complete

    # Copy intermediate results back to host for sum reduction.
    # For ultimate GPU performance, these sums could also be performed on the GPU using
    # `numba.cuda.reduce` or custom reduction kernels.
    numerator = numerator_parts_d.copy_to_host().sum()
    sum_sq_dev1 = dev1_parts_d.copy_to_host().sum()
    sum_sq_dev2 = dev2_parts_d.copy_to_host().sum()

    denominator = np.sqrt(sum_sq_dev1) * np.sqrt(sum_sq_dev2)

    if denominator == 0:
        return 0.0
    else:
        pcc = numerator / denominator
        if np.isnan(pcc):
            return 0.0
        return pcc

# Keep the original CPU-optimized version as a fallback or for explicit comparison
@jit(nopython=True)
def calculate_pcc_cpu(arr1: np.ndarray, arr2: np.ndarray) -> float:
    # 3. Flatten both input arrays into 1D vectors
    flat_arr1 = arr1.flatten()
    flat_arr2 = arr2.flatten()

    # Ensure both arrays have the same length after flattening
    if len(flat_arr1) != len(flat_arr2):
        return 0.0 # Or raise an error, depending on desired behavior

    # Calculate means
    mean1 = np.mean(flat_arr1)
    mean2 = np.mean(flat_arr2)

    # Calculate numerator (sum of (x_i - mean_x)(y_i - mean_y))
    numerator = np.sum((flat_arr1 - mean1) * (flat_arr2 - mean2))

    # Calculate denominators (sum of (x_i - mean_x)^2 and (y_i - mean_y)^2)
    sum_sq_dev1 = np.sum((flat_arr1 - mean1)**2)
    sum_sq_dev2 = np.sum((flat_arr2 - mean2)**2)

    # Handle potential division by zero (e.g., if one array has zero variance)
    denominator = np.sqrt(sum_sq_dev1) * np.sqrt(sum_sq_dev2)

    if denominator == 0:
        return 0.0 # If there's no variance in one or both, PCC is undefined, return 0
    else:
        pcc = numerator / denominator
        # Handle potential NaN if inputs contain NaNs and lead to NaNs in calculation
        if np.isnan(pcc):
            return 0.0
        return pcc

# Decide which version to use based on CUDA availability
try:
    if cuda.is_available():
        calculate_pcc = calculate_pcc_gpu
        print("Using GPU-optimized `calculate_pcc`.")
    else:
        calculate_pcc = calculate_pcc_cpu
        print("CUDA not available. Falling back to CPU-optimized `calculate_pcc`.")
except Exception as e:
    print(f"Error checking CUDA availability: {e}. Falling back to CPU-optimized `calculate_pcc`.")
    calculate_pcc = calculate_pcc_cpu

print("Libraries imported and `calculate_pcc` function defined and optimized with Numba (GPU or CPU fallback).")

start_time = time.time()

pcc_all_models_results = []

# Iterate through each model
for model_name, methods_data in all_loaded_data.items():
    method_names = list(methods_data.keys())

    # Iterate through all pairs of methods, including self-comparison and reversed pairs
    for method_a_name, method_b_name in itertools.product(method_names, repeat=2):
        # Get the set of image IDs common to both methods for this model
        common_image_ids = set(methods_data[method_a_name].keys()).intersection(methods_data[method_b_name].keys())

        for image_id in common_image_ids:
            map_a = methods_data[method_a_name][image_id]
            map_b = methods_data[method_b_name][image_id]

            # Ensure maps have the same shape before calculating PCC
            if map_a.shape == map_b.shape:
                pcc_value = calculate_pcc(map_a, map_b)
                pcc_all_models_results.append({
                    'model': model_name,
                    'image_id': image_id,
                    'method_a': method_a_name,
                    'method_b': method_b_name,
                    'PCC': pcc_value
                })
            else:
                print(f"⚠️ Warning: Saliency maps for {image_id} in model {model_name} have different shapes for {method_a_name} ({map_a.shape}) and {method_b_name} ({map_b.shape}). Skipping PCC for this pair.")

end_time = time.time()
total_calculation_time = end_time - start_time

print(f"Total PCC calculation time: {total_calculation_time:.2f} seconds")

# Convert results to DataFrame
df_pcc_all_models = pd.DataFrame(pcc_all_models_results)

# Save to CSV
output_csv_filename = '/content/drive/MyDrive/XAI_Project/results_multi_cam/pcc_all_models_full_matrix_results1.csv'
df_pcc_all_models.to_csv(output_csv_filename, index=False)

print(f"PCC full matrix results for all models and method comparisons saved to {output_csv_filename}")
print("First 5 rows of the results DataFrame:")
display(df_pcc_all_models.head())

In [ ]:
output_csv_filename = '/content/drive/MyDrive/XAI_Project/results_multi_cam/pcc_all_models_full_matrix_results1.csv'
df_pcc_all_models.to_csv(output_csv_filename, index=False)

print(f"PCC full matrix results for all models and method comparisons saved to {output_csv_filename}")
print("First 5 rows of the results DataFrame:")
display(df_pcc_all_models.head())

In [ ]:
import itertools
import time
import pandas as pd
import numpy as np
from skimage.metrics import structural_similarity as ssim # Import SSIM

# Function to calculate SSIM, expecting 2D arrays
def calculate_ssim(arr1: np.ndarray, arr2: np.ndarray) -> float:
    # Ensure both arrays have the same shape
    if arr1.shape != arr2.shape:
        return 0.0 # Or raise an error

    # Normalize arrays to a 0-1 range. This is crucial for SSIM to have a consistent data_range.
    # Saliency maps often have values that aren't strictly 0-1 initially.
    # We normalize each map independently based on its own min/max.
    # Add a small epsilon to avoid division by zero if max-min is zero (flat map)
    epsilon = 1e-8

    if arr1.max() - arr1.min() > epsilon:
        normalized_arr1 = (arr1 - arr1.min()) / (arr1.max() - arr1.min())
    else:
        normalized_arr1 = np.zeros_like(arr1) # If map is flat, normalize to all zeros

    if arr2.max() - arr2.min() > epsilon:
        normalized_arr2 = (arr2 - arr2.min()) / (arr2.max() - arr2.min())
    else:
        normalized_arr2 = np.zeros_like(arr2) # If map is flat, normalize to all zeros

    # Calculate SSIM. data_range=1.0 because we normalized to 0-1.
    # If the maps are 3D (e.g., RGB), channel_axis might be needed.
    # Assuming saliency maps are grayscale (2D) or will be treated as such.
    ssim_value = ssim(normalized_arr1, normalized_arr2, data_range=1.0)
    return ssim_value

print("SSIM function defined.")

start_time_ssim = time.time()

ssim_all_models_results = []

# Iterate through each model
for model_name, methods_data in all_loaded_data.items():
    method_names = list(methods_data.keys())

    # Iterate through all pairs of methods, including self-comparison and reversed pairs
    for method_a_name, method_b_name in itertools.product(method_names, repeat=2):
        # Get the set of image IDs common to both methods for this model
        common_image_ids = set(methods_data[method_a_name].keys()).intersection(methods_data[method_b_name].keys())

        for image_id in common_image_ids:
            map_a = methods_data[method_a_name][image_id]
            map_b = methods_data[method_b_name][image_id]

            # Ensure maps have the same shape before calculating SSIM
            if map_a.shape == map_b.shape:
                ssim_value = calculate_ssim(map_a, map_b)
                ssim_all_models_results.append({
                    'model': model_name,
                    'image_id': image_id,
                    'method_a': method_a_name,
                    'method_b': method_b_name,
                    'SSIM': ssim_value
                })
            else:
                print(f"⚠️ Warning: Saliency maps for {image_id} in model {model_name} have different shapes for {method_a_name} ({map_a.shape}) and {method_b_name} ({map_b.shape}). Skipping SSIM for this pair.")

end_time_ssim = time.time()
total_calculation_time_ssim = end_time_ssim - start_time_ssim

print(f"Total SSIM calculation time: {total_calculation_time_ssim:.2f} seconds")

# Convert results to DataFrame
df_ssim_all_models = pd.DataFrame(ssim_all_models_results)

# Save to CSV in the same directory as PCC results
output_csv_filename_ssim = '/content/drive/MyDrive/XAI_Project/results_multi_cam/ssim_all_models_full_matrix_results1.csv'
df_ssim_all_models.to_csv(output_csv_filename_ssim, index=False)

print(f"SSIM full matrix results for all models and method comparisons saved to {output_csv_filename_ssim}")
print("First 5 rows of the SSIM results DataFrame:")
display(df_ssim_all_models.head())

In [ ]:
output_csv_filename = '/content/drive/MyDrive/XAI_Project/results_multi_cam/results_all/pcc_all_models_full_matrix_results1.csv'
df_pcc_all_models.to_csv(output_csv_filename, index=False)

print(f"PCC full matrix results for all models and method comparisons saved to {output_csv_filename}")
print("First 5 rows of the results DataFrame:")
display(df_pcc_all_models.head())

In [ ]:
output_csv_filename_ssim = '/content/drive/MyDrive/XAI_Project/results_multi_cam/results_all/ssim_all_models_full_matrix_results1.csv'
df_ssim_all_models.to_csv(output_csv_filename_ssim, index=False)

print(f"SSIM full matrix results for all models and method comparisons saved to {output_csv_filename_ssim}")
print("First 5 rows of the SSIM results DataFrame:")
display(df_ssim_all_models.head())

In [ ]:
import time
import itertools
import pandas as pd
import numpy as np
from scipy.stats import spearmanr # Import Spearman's correlation

# Function to calculate Spearman's Rank Correlation Coefficient
def calculate_spearmanr(arr1: np.ndarray, arr2: np.ndarray) -> float:
    # Flatten both input arrays into 1D vectors
    flat_arr1 = arr1.flatten()
    flat_arr2 = arr2.flatten()

    # Ensure both arrays have the same length after flattening
    if len(flat_arr1) != len(flat_arr2):
        return 0.0 # Or raise an error

    # spearmanr returns (correlation_coefficient, p_value)
    corr, _ = spearmanr(flat_arr1, flat_arr2)

    # Handle potential NaN values (e.g., if one array has zero variance or all values are the same)
    if np.isnan(corr):
        return 0.0
    return corr

print("Spearman's Rank Correlation function defined.")

In [ ]:
import itertools
import time
import pandas as pd
import numpy as np
from scipy.stats import spearmanr # Import Spearman's correlation

# Function to calculate Spearman's Rank Correlation Coefficient
def calculate_spearmanr(arr1: np.ndarray, arr2: np.ndarray) -> float:
    # Flatten both input arrays into 1D vectors
    flat_arr1 = arr1.flatten()
    flat_arr2 = arr2.flatten()

    # Ensure both arrays have the same length after flattening
    if len(flat_arr1) != len(flat_arr2):
        return 0.0 # Or raise an error

    # spearmanr returns (correlation_coefficient, p_value)
    corr, _ = spearmanr(flat_arr1, flat_arr2)

    # Handle potential NaN values (e.g., if one array has zero variance or all values are the same)
    if np.isnan(corr):
        return 0.0
    return corr

print("Spearman's Rank Correlation function defined.")

start_time_spearmanr = time.time()

spearmanr_all_models_results = []

# Iterate through each model
for model_name, methods_data in all_loaded_data.items():
    method_names = list(methods_data.keys())

    # Iterate through all pairs of methods, including self-comparison and reversed pairs
    for method_a_name, method_b_name in itertools.product(method_names, repeat=2):
        # Get the set of image IDs common to both methods for this model
        common_image_ids = set(methods_data[method_a_name].keys()).intersection(methods_data[method_b_name].keys())

        for image_id in common_image_ids:
            map_a = methods_data[method_a_name][image_id]
            map_b = methods_data[method_b_name][image_id]

            # Ensure maps have the same shape before calculating Spearman's
            if map_a.shape == map_b.shape:
                spearmanr_value = calculate_spearmanr(map_a, map_b)
                spearmanr_all_models_results.append({
                    'model': model_name,
                    'image_id': image_id,
                    'method_a': method_a_name,
                    'method_b': method_b_name,
                    'SpearmanR': spearmanr_value
                })
            else:
                print(f"⚠️ Warning: Saliency maps for {image_id} in model {model_name} have different shapes for {method_a_name} ({map_a.shape}) and {method_b_name} ({map_b.shape}). Skipping SpearmanR for this pair.")

end_time_spearmanr = time.time()
total_calculation_time_spearmanr = end_time_spearmanr - start_time_spearmanr

print(f"Total Spearman's Rank Correlation calculation time: {total_calculation_time_spearmanr:.2f} seconds")

# Convert results to DataFrame
df_spearmanr_all_models = pd.DataFrame(spearmanr_all_models_results)

# Save to CSV in the same directory as PCC results
output_csv_filename_spearmanr = '/content/drive/MyDrive/XAI_Project/results_multi_cam/results_all/spearmanr_all_models_full_matrix_results1.csv'
df_spearmanr_all_models.to_csv(output_csv_filename_spearmanr, index=False)

print(f"Spearman's Rank Correlation full matrix results for all models and method comparisons saved to {output_csv_filename_spearmanr}")
print("First 5 rows of the Spearman's R results DataFrame:")
display(df_spearmanr_all_models.head())

In [ ]:
output_csv_filename_spearmanr = '/content/drive/MyDrive/XAI_Project/results_multi_cam/results_all/spearmanr_all_models_full_matrix_results1.csv'
df_spearmanr_all_models.to_csv(output_csv_filename_spearmanr, index=False)

print(f"Spearman's Rank Correlation full matrix results for all models and method comparisons saved to {output_csv_filename_spearmanr}")
print("First 5 rows of the Spearman's R results DataFrame:")
display(df_spearmanr_all_models.head())

In [ ]:
import time
import itertools
import pandas as pd
import numpy as np
from sklearn.metrics import matthews_corrcoef # Import MCC
from scipy.special import rel_entr # For KL Divergence

# Function to calculate Matthews Correlation Coefficient
def calculate_mcc(arr1: np.ndarray, arr2: np.ndarray) -> float:
    # Flatten both input arrays into 1D vectors
    flat_arr1 = arr1.flatten()
    flat_arr2 = arr2.flatten()

    # Ensure both arrays have the same length after flattening
    if len(flat_arr1) != len(flat_arr2):
        return 0.0 # Or raise an error

    # Binarize maps using their respective medians as thresholds
    # Handle constant arrays: if all values are the same, median is that value.
    # Using > median ensures there's at least some '1' if not all values are identical.

    if np.all(flat_arr1 == flat_arr1[0]): # Check if arr1 is constant
        bin_arr1 = np.zeros_like(flat_arr1, dtype=bool)
    else:
        median_arr1 = np.median(flat_arr1)
        bin_arr1 = flat_arr1 > median_arr1

    if np.all(flat_arr2 == flat_arr2[0]): # Check if arr2 is constant
        bin_arr2 = np.zeros_like(flat_arr2, dtype=bool)
    else:
        median_arr2 = np.median(flat_arr2)
        bin_arr2 = flat_arr2 > median_arr2

    # matthews_corrcoef raises ValueError if input contains only one class
    # or if the confusion matrix leads to division by zero (e.g., all True or all False)
    if len(np.unique(bin_arr1)) < 2 or len(np.unique(bin_arr2)) < 2:
        return 0.0 # MCC is undefined in such cases, return 0.0 as a neutral value

    try:
        mcc = matthews_corrcoef(bin_arr1, bin_arr2)
        return mcc
    except ValueError: # Catch cases where MCC cannot be computed (e.g., all predictions are the same)
        return 0.0 # Return 0.0 for undefined MCC

# Function to calculate KL Divergence
def calculate_kl_divergence(arr1: np.ndarray, arr2: np.ndarray) -> float:
    # Flatten both input arrays into 1D vectors
    p = arr1.flatten()
    q = arr2.flatten()

    # Ensure both arrays have the same length after flattening
    if len(p) != len(q):
        return np.nan # Or raise an error

    # Normalize to form probability distributions
    # Add a small epsilon to avoid division by zero when normalizing or taking log
    epsilon = 1e-10
    p = p + epsilon
    q = q + epsilon

    p_sum = np.sum(p)
    q_sum = np.sum(q)

    if p_sum == 0 or q_sum == 0:
        return np.nan # Cannot calculate KL if one distribution sums to zero

    p_normalized = p / p_sum
    q_normalized = q / q_sum

    # Calculate KL Divergence using scipy.special.rel_entr
    # rel_entr(p, q) calculates p * log(p/q) element-wise
    kl_div = np.sum(rel_entr(p_normalized, q_normalized))
    return kl_div

print("Matthews Correlation Coefficient (MCC) and Kullback-Leibler (KL) Divergence functions defined.")

In [ ]:
start_time_mcc = time.time()

mcc_all_models_results = []
kl_div_all_models_results = []

# Iterate through each model
for model_name, methods_data in all_loaded_data.items():
    method_names = list(methods_data.keys())

    # Iterate through all pairs of methods, including self-comparison and reversed pairs
    for method_a_name, method_b_name in itertools.product(method_names, repeat=2):
        # Get the set of image IDs common to both methods for this model
        common_image_ids = set(methods_data[method_a_name].keys()).intersection(methods_data[method_b_name].keys())

        for image_id in common_image_ids:
            map_a = methods_data[method_a_name][image_id]
            map_b = methods_data[method_b_name][image_id]

            # Ensure maps have the same shape before calculating metrics
            if map_a.shape == map_b.shape:
                # Calculate MCC
                mcc_value = calculate_mcc(map_a, map_b)
                mcc_all_models_results.append({
                    'model': model_name,
                    'image_id': image_id,
                    'method_a': method_a_name,
                    'method_b': method_b_name,
                    'MCC': mcc_value
                })

                # Calculate KL Divergence
                kl_div_value = calculate_kl_divergence(map_a, map_b)
                kl_div_all_models_results.append({
                    'model': model_name,
                    'image_id': image_id,
                    'method_a': method_a_name,
                    'method_b': method_b_name,
                    'KL_Divergence': kl_div_value
                })
            else:
                print(f"⚠️ Warning: Saliency maps for {image_id} in model {model_name} have different shapes for {method_a_name} ({map_a.shape}) and {method_b_name} ({map_b.shape}). Skipping MCC and KL for this pair.")

end_time_mcc = time.time()
total_calculation_time_mcc = end_time_mcc - start_time_mcc

print(f"Total MCC and KL Divergence calculation time: {total_calculation_time_mcc:.2f} seconds")

# Convert MCC results to DataFrame and save to CSV
df_mcc_all_models = pd.DataFrame(mcc_all_models_results)
output_csv_filename_mcc = '/content/drive/MyDrive/XAI_Project/results_multi_cam/results_all/mcc_all_models_full_matrix_results1.csv'
df_mcc_all_models.to_csv(output_csv_filename_mcc, index=False)
print(f"MCC results for all models and method comparisons saved to {output_csv_filename_mcc}")
print("First 5 rows of the MCC results DataFrame:")
display(df_mcc_all_models.head())

# Convert KL Divergence results to DataFrame and save to CSV
df_kl_div_all_models = pd.DataFrame(kl_div_all_models_results)
output_csv_filename_kl = '/content/drive/MyDrive/XAI_Project/results_multi_cam/results_all/kl_divergence_all_models_full_matrix_results1.csv'
df_kl_div_all_models.to_csv(output_csv_filename_kl, index=False)
print(f"KL Divergence results for all models and method comparisons saved to {output_csv_filename_kl}")
print("First 5 rows of the KL Divergence results DataFrame:")
display(df_kl_div_all_models.head())

In [ ]:

import time
import itertools
import pandas as pd
import numpy as np
from scipy.spatial.distance import jensenshannon

# Function to calculate Jensen-Shannon Divergence
def calculate_jsd(arr1: np.ndarray, arr2: np.ndarray) -> float:
    # Flatten both input arrays into 1D vectors
    P = arr1.flatten()
    Q = arr2.flatten()

    # Ensure both arrays have the same length after flattening
    if len(P) != len(Q):
        return np.nan # Or raise an error

    # Normalize to form probability distributions
    # Add a small epsilon to avoid division by zero when normalizing or taking log
    epsilon = 1e-10
    P = P + epsilon
    Q = Q + epsilon

    P_sum = np.sum(P)
    Q_sum = np.sum(Q)

    if P_sum == 0 or Q_sum == 0:
        return np.nan # Cannot calculate JSD if one distribution sums to zero

    P_normalized = P / P_sum
    Q_normalized = Q / Q_sum

    # jensenshannon returns the square root of JSD, so square it for JSD
    jsd_value = jensenshannon(P_normalized, Q_normalized)**2
    return jsd_value

print("Jensen-Shannon Divergence function defined.")

start_time_jsd = time.time()

jsd_all_models_results = []

# Iterate through each model
for model_name, methods_data in all_loaded_data.items():
    method_names = list(methods_data.keys())

    # Iterate through all pairs of methods, including self-comparison and reversed pairs
    for method_a_name, method_b_name in itertools.product(method_names, repeat=2):
        # Get the set of image IDs common to both methods for this model
        common_image_ids = set(methods_data[method_a_name].keys()).intersection(methods_data[method_b_name].keys())

        for image_id in common_image_ids:
            map_a = methods_data[method_a_name][image_id]
            map_b = methods_data[method_b_name][image_id]

            # Ensure maps have the same shape before calculating JSD
            if map_a.shape == map_b.shape:
                jsd_value = calculate_jsd(map_a, map_b)
                jsd_all_models_results.append({
                    'model': model_name,
                    'image_id': image_id,
                    'method_a': method_a_name,
                    'method_b': method_b_name,
                    'JSD': jsd_value
                })
            else:
                print(f"⚠️ Warning: Saliency maps for {image_id} in model {model_name} have different shapes for {method_a_name} ({map_a.shape}) and {method_b_name} ({map_b.shape}). Skipping JSD for this pair.")

end_time_jsd = time.time()
total_calculation_time_jsd = end_time_jsd - start_time_jsd

print(f"Total JSD calculation time: {total_calculation_time_jsd:.2f} seconds")

# Convert results to DataFrame
df_jsd_all_models = pd.DataFrame(jsd_all_models_results)

# Save to CSV in the same directory as other results
output_csv_filename_jsd = '/content/drive/MyDrive/XAI_Project/results_multi_cam/results_all/jsd_all_models_full_matrix_results1.csv'
df_jsd_all_models.to_csv(output_csv_filename_jsd, index=False)

print(f"JSD full matrix results for all models and method comparisons saved to {output_csv_filename_jsd}")
print("First 5 rows of the JSD results DataFrame:")
#display(df_jsd_all_models.head())


In [ ]:
import time
import itertools
import pandas as pd
import numpy as np
from scipy.spatial.distance import jensenshannon

# Function to calculate Jensen-Shannon Divergence
def calculate_jsd(arr1: np.ndarray, arr2: np.ndarray) -> float:
    # Flatten both input arrays into 1D vectors
    P = arr1.flatten()
    Q = arr2.flatten()

    # Ensure both arrays have the same length after flattening
    if len(P) != len(Q):
        return np.nan # Or raise an error

    # Add a small epsilon to avoid division by zero errors or log(0) when normalizing or taking log
    # Also, ensure values are non-negative before normalization
    P = np.maximum(P, 0) + 1e-10
    Q = np.maximum(Q, 0) + 1e-10

    P_sum = np.sum(P)
    Q_sum = np.sum(Q)

    if P_sum == 0 or Q_sum == 0:
        return np.nan # Cannot calculate JSD if one distribution sums to zero

    P_normalized = P / P_sum
    Q_normalized = Q / Q_sum

    # jensenshannon returns the square root of JSD, so square it for JSD
    jsd_value = jensenshannon(P_normalized, Q_normalized)**2
    return jsd_value

print("Jensen-Shannon Divergence function defined.")

# --- Adapted Insertion and Deletion Tests for Saliency Maps ---

def insertion_test_saliency_maps(map_A: np.ndarray, map_B: np.ndarray, eps: float = 1e-3) -> float:
    """
    Calculates JSD between a reference saliency map (map_A) and a perturbed saliency map (map_B_ins).
    map_B_ins is created by adding `eps` to the first element of the flattened map_B.
    """
    if map_A.shape != map_B.shape:
        return np.nan # Or handle appropriately

    map_B_ins = map_B.copy()
    # Perturb the first element of the flattened array to work with any shape
    map_B_ins.flat[0] += eps

    return calculate_jsd(map_A, map_B_ins)

def deletion_test_saliency_maps(map_A: np.ndarray, map_B: np.ndarray) -> float:
    """
    Calculates JSD between a reference saliency map (map_A) and a perturbed saliency map (map_B_del).
    map_B_del is created by setting the first element of the flattened map_B to 0.
    """
    if map_A.shape != map_B.shape:
        return np.nan # Or handle appropriately

    map_B_del = map_B.copy()
    # Perturb the first element of the flattened array to work with any shape
    map_B_del.flat[0] = 0.0

    return calculate_jsd(map_A, map_B_del)

print("Insertion and Deletion test functions for saliency maps defined.")

start_time_tests = time.time()

insertion_results = []
deletion_results = []

# Iterate through each model
for model_name, methods_data in all_loaded_data.items():
    method_names = list(methods_data.keys())

    # Iterate through all pairs of methods (A as reference, B as perturbed)
    for method_a_name, method_b_name in itertools.product(method_names, repeat=2):
        common_image_ids = set(methods_data[method_a_name].keys()).intersection(methods_data[method_b_name].keys())

        for image_id in common_image_ids:
            map_a = methods_data[method_a_name][image_id]
            map_b = methods_data[method_b_name][image_id]

            # Ensure maps have the same shape before calculating metrics
            if map_a.shape == map_b.shape:
                # Calculate Insertion Test
                insertion_val = insertion_test_saliency_maps(map_a, map_b)
                insertion_results.append({
                    'model': model_name,
                    'image_id': image_id,
                    'reference_method': method_a_name,
                    'perturbed_method': method_b_name,
                    'JSD_Insertion': insertion_val
                })

                # Calculate Deletion Test
                deletion_val = deletion_test_saliency_maps(map_a, map_b)
                deletion_results.append({
                    'model': model_name,
                    'image_id': image_id,
                    'reference_method': method_a_name,
                    'perturbed_method': method_b_name,
                    'JSD_Deletion': deletion_val
                })
            else:
                print(f"⚠️ Warning: Saliency maps for {image_id} in model {model_name} have different shapes for {method_a_name} ({map_a.shape}) and {method_b_name} ({map_b.shape}). Skipping Insertion/Deletion for this pair.")

end_time_tests = time.time()
total_calculation_time_tests = end_time_tests - start_time_tests

print(f"Total Insertion and Deletion tests calculation time: {total_calculation_time_tests:.2f} seconds")

# Convert Insertion results to DataFrame and save to CSV
df_insertion_tests = pd.DataFrame(insertion_results)
output_csv_filename_insertion = '/content/drive/MyDrive/XAI_Project/results_multi_cam/results_all/jsd_insertion_test_results1.csv'
df_insertion_tests.to_csv(output_csv_filename_insertion, index=False)
print(f"Insertion test results for all models and method comparisons saved to {output_csv_filename_insertion}")
print("First 5 rows of the Insertion results DataFrame:")
display(df_insertion_tests.head())

# Convert Deletion results to DataFrame and save to CSV
df_deletion_tests = pd.DataFrame(deletion_results)
output_csv_filename_deletion = '/content/drive/MyDrive/XAI_Project/results_multi_cam/results_all/jsd_deletion_test_results1.csv'
df_deletion_tests.to_csv(output_csv_filename_deletion, index=False)
print(f"Deletion test results for all models and method comparisons saved to {output_csv_filename_deletion}")
print("First 5 rows of the Deletion results DataFrame:")
display(df_deletion_tests.head())
